In [ ]:
# Set up
from google.colab import drive
drive.mount('/content/drive')
!pip install mne --quiet
import os
import mne
import glob
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from scipy.fft import fft, ifft
from tqdm import tqdm
import scipy.io.wavfile as wav
from mne.time_frequency import tfr_array_morlet

In [ ]:
# Morlet Wavelet Analysis

In [ ]:
# METHOD 1

root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/'
wav_files = sorted([
    f for f in glob.glob(os.path.join(root_dir, '*.wav'))
    if f.endswith('_N.wav') or f.endswith('_E.wav')
])

freqs = np.linspace(100, 7000, 100)
n_cycles = 3
smooth_ms = 5

for wav_path in wav_files:
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data))  # normalize
    t = np.arange(len(data)) / fs

    # Reshape for MNE (1 "epoch", 1 "channel", n_times)
    data_reshaped = data[np.newaxis, np.newaxis, :]

    # Compute Morlet-based TFR amplitude
    # Use MNE’s function to convolve audio with Morlet wavelets for each frequency.
    # Get the power (amplitude squared) of the result.
    power = tfr_array_morlet(data_reshaped, sfreq=fs, freqs=freqs,
                             n_cycles=n_cycles, output='power', zero_mean=True) # shape: (1, 1, n_freqs, n_times).

    # Average amplitude across frequencies
    # Compute broadband envelope:
    # Take square root (to get amplitude).
    # Average across frequencies → final broadband envelope (1D).
    envelope_wavelet_avg = np.mean(np.sqrt(power[0, 0, :, :]), axis=0)

    # Smooth envelope with a Gaussian filter
    sigma = (smooth_ms / 1000.0) * fs
    envelope_smooth = gaussian_filter1d(envelope_wavelet_avg, sigma)
    # envelope_smooth = envelope_wavelet_avg

    # Compute derivative Calculate first derivative of smoothed envelope → shows rate of amplitude change.
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # Plot
    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform')
    plt.plot(t, envelope_smooth, linewidth=2, label='Wavelet Envelope', color='orange')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.title('Wavelet-based Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='green', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Wavelet Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# METHOD 2

### STEP 1 — Extract envelope from audio

# Directory containing .wav files
stimuli_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli'
wav_files = [f for f in os.listdir(stimuli_dir) if f.endswith('.wav')]

# Process all stimuli
stimuli_envelopes = {}
for wav_file in wav_files:
    path = os.path.join(stimuli_dir, wav_file)

    # Load audio
    audio, sr_audio = librosa.load(path, sr=None)

    # Hilbert transform to get envelope
    analytic = hilbert(audio)
    envelope = np.abs(analytic)

    # Smooth envelope (optional)
    envelope_smooth = gaussian_filter1d(envelope, sigma=100)

    # Normalize envelope
    envelope_norm = envelope_smooth / np.max(envelope_smooth)

    # Store for later
    stimuli_envelopes[wav_file] = (envelope_norm, sr_audio)

    # Plot example
    plt.figure(figsize=(10, 3))
    plt.plot(audio/np.max(np.abs(audio)), label='Waveform')
    plt.plot(envelope_norm, color='orange', label='Envelope')
    plt.title(f'Envelope – {wav_file}')
    plt.legend()
    plt.show()


### STEP 2 — Resample envelope to EEG sampling rate

# EEG sampling rate
sr_eeg = 512

# Example: resample 1st stimulus
example_file = wav_files[0]
envelope, sr_audio = stimuli_envelopes[example_file]
envelope_resampled = librosa.resample(envelope, orig_sr=sr_audio, target_sr=sr_eeg)

# Make sure envelope length matches EEG length for alignment
# (this assumes you know the duration matches EEG segment)


### STEP 3 — Extract time-frequency features from EEG using Morlet

# Load your EEG data (example: 88 channels, 10 sec recording)
EEG = np.random.randn(81, sr_eeg * 10)  # Replace this with your real EEG

# Morlet parameters
freqs = np.logspace(np.log10(100), np.log10(10000), 200)
n_cycles = 6

n_channels, n_samples = EEG.shape
n_frequencies = len(freqs)
power = np.zeros((n_channels, n_frequencies, n_samples))

# FFT convolution
n_conv = n_samples + len(EEG[0,:]) - 1
EEG_fft = fft(EEG, n=n_conv, axis=1)

for f_idx, f in tqdm(enumerate(freqs), total=len(freqs), desc="Morlet transform"):

    wavetime = np.arange(-1, 1, 1/sr_eeg)
    sigma = n_cycles / (2 * np.pi * f)
    wavelet = np.exp(2j * np.pi * f * wavetime) * np.exp(-wavetime**2 / (2 * sigma**2))
    wavelet = wavelet / np.sqrt(sigma * np.sqrt(np.pi))

    wavelet_fft = fft(wavelet, n=n_conv)

    for ch in range(n_channels):
        convolution_result = ifft(EEG_fft[ch, :] * wavelet_fft)
        convolution_result = convolution_result[:n_samples]
        power[ch, f_idx, :] = np.abs(convolution_result) ** 2


### STEP 4 — Align EEG power & envelope

# Now your EEG power and envelope_resampled are both time-aligned
# You can analyze e.g.:
# - Correlations
# - TRF models
# - Decoding
# - Phase-locking

# Example: plot time-frequency power for 1 channel
plt.figure(figsize=(10, 5))
plt.imshow(power[0,:,:], aspect='auto',
           extent=[0, n_samples/sr_eeg, freqs[0], freqs[-1]],
           origin='lower', cmap='jet')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Time-Frequency Power (Channel 0)')
plt.colorbar(label='Power')
plt.show()


In [ ]:
# Try to handle both cases

stimuli_envelope_derivatives = {}

for wav_file in wav_files:
    data = stimuli_envelopes[wav_file]

    # Check if data is tuple or array
    if isinstance(data, tuple):
        envelope_norm, sr_audio = data
    else:
        envelope_norm = data
        sr_audio = 44100  # <-- or whatever your default sample rate is

    dt = 1 / sr_audio
    envelope_derivative = np.gradient(envelope_norm, dt)

    stimuli_envelope_derivatives[wav_file] = (envelope_derivative, sr_audio)

    # Plot
    plt.figure(figsize=(10, 3))
    plt.plot(envelope_norm, label='Envelope')
    plt.plot(envelope_derivative, label='Derivative', color='red')
    plt.title(f'Derivative – {wav_file}')
    plt.legend()
    plt.show()



In [ ]:
plt.plot(envelope)
plt.title("Envelope before smoothing")
plt.show()


In [ ]:
plt.plot(envelope_smooth)
plt.title("Envelope after smoothing")
plt.show()


In [ ]:
print("Resampled envelope shape:", envelope_resampled.shape)
print("Resampled envelope max:", np.max(envelope_resampled))

plt.plot(envelope_resampled)
plt.title("Envelope after resampling")
plt.show()


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from scipy.ndimage import gaussian_filter1d
from mne.time_frequency import tfr_array_morlet

# ---------------------
# PARAMETERS
# ---------------------
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/'
wav_files = sorted([
    f for f in glob.glob(os.path.join(root_dir, '*.wav'))
    if f.endswith('_N.wav') or f.endswith('_E.wav')
])

# Improved parameters:
freqs = np.logspace(np.log10(80), np.log10(5000), 80)  # log-spaced frequencies
n_cycles = 5                                            # more stable envelope
smooth_ms = 10                                          # slightly more smoothing

# ---------------------
# PROCESSING LOOP
# ---------------------
for wav_path in wav_files:
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data))  # normalize to -1...1
    t = np.arange(len(data)) / fs

    # Reshape for MNE (1 "epoch", 1 "channel", n_times)
    data_reshaped = data[np.newaxis, np.newaxis, :]

    # Compute Morlet-based TFR amplitude
    power = tfr_array_morlet(data_reshaped, sfreq=fs, freqs=freqs,
                              n_cycles=n_cycles, output='power', zero_mean=True)

    # Compute broadband envelope: average across frequencies
    envelope_wavelet_avg = np.mean(np.sqrt(power[0, 0, :, :]), axis=0)

    # Smooth envelope
    sigma = (smooth_ms / 1000.0) * fs
    envelope_smooth = gaussian_filter1d(envelope_wavelet_avg, sigma)

    # Compute first derivative
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # ---- PLOTTING ----

    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform', color='steelblue')
    plt.plot(t, envelope_smooth, linewidth=2, label='Wavelet Envelope', color='orange')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.title('Wavelet-based Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='green', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Wavelet Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from scipy.ndimage import gaussian_filter1d
from mne.time_frequency import tfr_array_morlet

# ---------------------
# PARAMETERS
# ---------------------
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/'
wav_files = sorted([
    f for f in glob.glob(os.path.join(root_dir, '*.wav'))
    if f.endswith('_N.wav') or f.endswith('_E.wav')
])

# Improved parameters:
freqs = np.logspace(np.log10(80), np.log10(5000), 80)  # log-spaced frequencies
n_cycles = 5                                            # more stable envelope
smooth_ms = 10                                          # slightly more smoothing

# ---------------------
# PROCESSING LOOP
# ---------------------
for wav_path in wav_files:
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data))  # normalize to -1...1
    t = np.arange(len(data)) / fs

    # Reshape for MNE (1 "epoch", 1 "channel", n_times)
    data_reshaped = data[np.newaxis, np.newaxis, :]

    # Compute Morlet-based TFR amplitude
    power = tfr_array_morlet(data_reshaped, sfreq=fs, freqs=freqs,
                              n_cycles=n_cycles, output='power', zero_mean=True)

    # Compute broadband envelope: average across frequencies
    envelope_wavelet_avg = np.mean(np.sqrt(power[0, 0, :, :]), axis=0)

    # Smooth envelope
    sigma = (smooth_ms / 1000.0) * fs
    envelope_smooth = gaussian_filter1d(envelope_wavelet_avg, sigma)

    # Compute first derivative
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # ---- PLOTTING ----

    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform', color='steelblue')
    plt.plot(t, envelope_smooth, linewidth=2, label='Wavelet Envelope', color='orange')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.title('Wavelet-based Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='green', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Wavelet Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()
